In [1]:
import awkward as ak
import numba
import numpy as np
import pandas as pd
import awkward as ak
import vector
import numba
import torch
from torch import nn 
from torch.optim import RAdam
from scipy.special import softmax
import h5py
import yaml
import argparse

import seaborn as sns

from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, auc
import onnxruntime    # to inference ONNX models, we use the ONNX Runtime
vector.register_numba()
vector.register_awkward()
import os

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import mplhep as hep
hep.style.use(hep.style.ROOT)

from importlib import reload
from sklearn.preprocessing import QuantileTransformer
from copy import deepcopy
import numpy as np
from scipy.stats import norm
from sklearn.base import BaseEstimator, TransformerMixin
from multiprocessing import Pool, cpu_count
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

In [2]:
dctr_input = "/eos/home-g/gbonomel/tth/dctr/training_datasets/btag_shapes/for_inference/dataset_dctr_4j3b_withsignal_allyears_baseline.parquet"
spanet_model = "/eos/home-g/gbonomel/tth/spanet/models/singlewp_dctr_leptonveto_medium/spanet_evaluation/spanet_onnx_inference.parquet"
event_f = "/eos/home-g/gbonomel/tth/spanet/models/training_dec25/event.yaml"
dctr_model = "/eos/home-g/gbonomel/tth/dctr/plot_input_variables/model_adv_epoch40.pth"
save_folder = "/eos/home-g/gbonomel/tth/dctr/plot_input_variables"

# ensure save folder exists
os.makedirs(save_folder, exist_ok=True)

CMAP_10 = [
    "#3f90da",
    "#ffa90e",
    "#bd1f01",
    "#94a4a2",
    "#832db6",
    "#a96b59",
    "#e76300",
    "#b9ac70",
    "#717581",
    "#92dadd",
]

In [3]:
labels = {
    "tthbb": 1,
    "ttbb": 2,
    "ttcc": 3,
    "ttlf": 4,
    "ttv": 5,
    "singletop": 6
}

In [4]:
tthbb_transformed_min = 0.5
tthbb_transformed_max = 0.8
ttlf_max = 0.5

In [5]:
def get_cr_mask(scores, tthbb_transformed_min, tthbb_transformed_max, ttlf_max):
    mask = (
        (scores["tthbb"] >= tthbb_transformed_min) &
        (scores["tthbb"] < tthbb_transformed_max) &
        (scores["ttlf"] < ttlf_max)
    )
    return mask 

def get_sr_mask(scores, tthbb_transformed_min, tthbb_transformed_max, ttlf_max):
    mask = (
        (scores["tthbb"] >= tthbb_transformed_max) &
        (scores["ttlf"] < ttlf_max)
    )
    return mask

In [6]:
class WeightedQuantileTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_quantiles=1000, output_distribution='normal'):
        self.n_quantiles = n_quantiles
        self.output_distribution = output_distribution

    def save(self, filename):
        np.savez(
            filename,
            quantiles=self.quantiles_,
            reference_quantiles=self.reference_quantiles_,
            n_quantiles=self.n_quantiles,
            output_distribution=self.output_distribution,
        )

    def load(self, filename):
        data = np.load(filename)
        self.quantiles_ = data["quantiles"]
        self.reference_quantiles_ = data["reference_quantiles"]
        self.n_quantiles = int(data["n_quantiles"])
        self.output_distribution = str(data["output_distribution"])
    
    def _weighted_quantiles(self, X, weights):
        # Calculate weighted quantiles
        sorted_indices = np.argsort(X)
        X_sorted = X[sorted_indices]
        weights_sorted = weights[sorted_indices]
        cum_weights = np.cumsum(weights_sorted) / np.sum(weights_sorted)
        
        # Interpolate to get quantiles
        quantiles = np.interp(np.linspace(0, 1, self.n_quantiles), cum_weights, X_sorted)
        return quantiles
    
    def fit(self, X, y=None, sample_weight=None):
        if sample_weight is None:
            raise ValueError("Sample weights must be provided.")
        
        self.quantiles_ = self._weighted_quantiles(X, sample_weight)
        
        if self.output_distribution == 'normal':
            self.reference_quantiles_ = norm.ppf(np.linspace(0, 1, self.n_quantiles))
        elif self.output_distribution == 'uniform':
            self.reference_quantiles_ = np.linspace(0, 1, self.n_quantiles)
        else:
            raise ValueError(f"Unknown output distribution '{self.output_distribution}'.")
        
        return self
    
    def transform(self, X):
        # Interpolate based on weighted quantiles
        transformed_X = np.interp(X, self.quantiles_, self.reference_quantiles_)
        return transformed_X

In [7]:
# Simple DNN Class Definition (must match training)
class DCTRNet(nn.Module):
    def __init__(self, input_dim, hidden_net=[64,64,64], hidden_net_classifier=[64,64], hidden_net_adv=[64,64]):
        super(DCTRNet, self).__init__()
        layers = []
        for h_dim in hidden_net:
            layers.append(nn.Linear(input_dim, h_dim))
            layers.append(nn.ReLU())
            input_dim = h_dim

        layers.append(nn.Linear(input_dim, input_dim))
        self.pre_model = nn.Sequential(*layers)
        self.features_dim = hidden_net[-1]

        input_dim = self.features_dim
        layers_classifier = []
        for h_dim in hidden_net_classifier:
            layers_classifier.append(nn.Linear(input_dim, h_dim))
            layers_classifier.append(nn.ReLU())
            input_dim = h_dim
        layers_classifier.append(nn.Linear(input_dim, 1))
        self.classifier = nn.Sequential(*layers_classifier)

        input_dim = self.features_dim
        adv_layers = []
        for h_dim in hidden_net_adv:
            adv_layers.append(nn.Linear(input_dim, h_dim))
            adv_layers.append(nn.ReLU())
            input_dim = h_dim
        adv_layers.append(nn.Linear(input_dim, 1))
        self.model_adv = nn.Sequential(*adv_layers)
    
    def forward(self, x, return_features=False, run_adv=False):
        features = self.pre_model(x)
        if run_adv:
            adv_output = self.model_adv(features)
            if return_features:
                return adv_output, features
            return adv_output
        else:
            if return_features:
                return self.classifier(features), features
            return self.classifier(features)

Open the spanet output to get spanet scores and weights. Add the spanet scores as a field to the DCTR parquet data frame

In [8]:
# load the dctr model and dctr inputs
df = ak.from_parquet(dctr_input)
spanet_out = ak.from_parquet(spanet_model)

In [9]:
# extract weights by event
weights = ak.to_numpy(df.event.weight)

In [10]:
# true labels from dctr input to use as mask by sample
true_labels = {}
for label in labels:
    true_labels[label] = ak.to_numpy(df[f"{label}"]).astype(bool)
true_labels

{'tthbb': array([False, False, False, ..., False, False, False]),
 'ttbb': array([False, False, False, ..., False,  True, False]),
 'ttcc': array([False, False, False, ..., False, False, False]),
 'ttlf': array([ True,  True,  True, ...,  True, False,  True]),
 'ttv': array([False, False, False, ..., False, False, False]),
 'singletop': array([False, False, False, ..., False, False, False])}

In [11]:
# convert spanet output into numpy arrays without masked arrays
pred_labels = {}
for key in spanet_out.fields:
    pred_labels[key] = ak.to_numpy(ak.fill_none(spanet_out[key], 0.0))
pred_labels

{'tthbb': array([0.01677565, 0.02462359, 0.10016861, ..., 0.02080909, 0.26315096,
        0.08540645]),
 'ttbb': array([0.04531126, 0.07313527, 0.09297699, ..., 0.04560841, 0.34171772,
        0.14383087]),
 'ttcc': array([0.10255951, 0.17910963, 0.10449398, ..., 0.23260058, 0.02688115,
        0.26112619]),
 'ttlf': array([0.24880981, 0.26536328, 0.104104  , ..., 0.5371269 , 0.02242004,
        0.24478647]),
 'ttv': array([0.21260256, 0.20786221, 0.51113534, ..., 0.02679482, 0.20191857,
        0.02492064]),
 'singletop': array([0.37394121, 0.24990599, 0.08712104, ..., 0.13706021, 0.14391163,
        0.23992936]),
 'weight': array([-0.05854146, -0.06429805, -0.02314291, ..., -0.01669056,
         0.0378209 , -0.14732438])}

In [12]:
# transform the tthbb score with quantile transformation
is_tthbb = true_labels["tthbb"]
score_tthbb = ak.to_numpy(pred_labels["tthbb"][is_tthbb])
weights_tthbb = weights[is_tthbb]
pred_labels_transformed = deepcopy(pred_labels)
transformer = WeightedQuantileTransformer(n_quantiles=10000, output_distribution='uniform')
transformer.fit(score_tthbb, sample_weight=weights_tthbb)
pred_labels_transformed["tthbb"] = transformer.transform(pred_labels["tthbb"])

In [13]:
# add the spanet output to the df 
spanet_fields = {key: pred_labels_transformed[key] for key in pred_labels_transformed.keys()}
df = ak.with_field(df, ak.zip(spanet_fields), "spanet_output")
df.fields

['JetGood',
 'LeptonGood',
 'PuppiMET',
 'events',
 'tthbb',
 'ttbb',
 'ttcc',
 'ttlf',
 'ttv',
 'singletop',
 'vv',
 'vjets',
 'data',
 'dctr',
 'metadata',
 'BJetGood',
 'event',
 'spanet_output']

In [14]:
df.spanet_output.fields

['tthbb', 'ttbb', 'ttcc', 'ttlf', 'ttv', 'singletop', 'weight']

In [15]:
df # contains all regions 

<Array [{JetGood: [, ... weight: -0.147}}] type='9464135 * {"JetGood": var * {"p...'>

In [16]:
# dctr input variables
data = []
bjet_pt_3 = df.BJetGood.pt[:,2]
bjet_eta_3 = df.BJetGood.eta[:,2]
for f in df.events.fields:
    data.append(ak.to_numpy(df.events[f]))
data.append(ak.to_numpy(bjet_pt_3))
data.append(ak.to_numpy(bjet_eta_3))
dctr_inputs = np.stack(data).T

In [17]:
dctr_inputs.shape # contains all regions

(9464135, 12)

In [18]:
# std normalize the dctr inputs (mean 0 std 1) before converting to tensor
scaler = StandardScaler()
x_scaled = scaler.fit_transform(dctr_inputs.astype(np.float32))
x_tensor = torch.from_numpy(x_scaled)

In [19]:
# Initialize the model with the same architecture as training
model = DCTRNet(input_dim=x_tensor.shape[1], hidden_net=[64,32], 
                         hidden_net_classifier=[32,32], hidden_net_adv=[16,16]).to("cuda")

# load already existing model
model_path = dctr_model 
if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location=torch.device('cuda'))
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
         model.load_state_dict(checkpoint['state_dict']) # incase it was saved wrapped
    else:
         model.load_state_dict(checkpoint)
    print(f"Loaded model from {model_path}")
else:
    print(f"Model file not found at {model_path}")

model.eval()

with torch.no_grad():
   inputs = x_tensor.to('cuda')
   outputs = model(inputs)
   # Apply sigmoid for probability output
   predictions = torch.sigmoid(outputs).squeeze().cpu()

/tmp/gbonomel/ipykernel_3646942/3311278993.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=torch.device('cuda'))


Loaded model from /eos/home-g/gbonomel/tth/dctr/plot_input_variables/model_adv_epoch40.pth


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.26 GiB. GPU 0 has a total capacity of 14.56 GiB of which 488.75 MiB is free. Process 3614131 has 13.56 GiB memory in use. Including non-PyTorch memory, this process has 538.00 MiB memory in use. Of the allocated memory 434.06 MiB is allocated by PyTorch, and 1.94 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
dctr_score = predictions.cpu().detach().numpy()
dctr_w_ttbb = dctr_score / (1 - dctr_score)

In [77]:
# Split events in 33% quantiles of the DCTR weight
q = [1./3, 2./3] 
weight_cuts_dict = {}
for nj in range(4, 8):
    if nj == 7:
        mask_nj = ak.to_numpy(njet >= nj)
    else:
        mask_nj = ak.to_numpy(njet == nj)
    
    # Mask for ttbb events in ttbb_CR with specific njet
    mask_ttbb_nj = (~is_data) & mask_nj & mask_ttbb_CR
    
    # Get dctr weights for these events using full-size array
    weights_by_njet = dctr_weight_full[mask_ttbb_nj]
    
    # Calculate quantiles
    w_lo, w_hi = np.quantile(weights_by_njet, q)
    
    if nj == 7:
        weight_cuts_dict[f"njet>={nj}"] = [(0, w_lo), (w_lo, w_hi), (w_hi, 3)]
    else:
        weight_cuts_dict[f"njet={nj}"] = [(0, w_lo), (w_lo, w_hi), (w_hi, 3)]

weight_cuts_dict

{'njet=4': [(0, 0.9353789289792379),
  (0.9353789289792379, 1.0436167319615681),
  (1.0436167319615681, 3)],
 'njet=5': [(0, 0.9189627170562744),
  (0.9189627170562744, 1.0264906485875447),
  (1.0264906485875447, 3)],
 'njet=6': [(0, 0.9219486117362976),
  (0.9219486117362976, 1.0322153568267822),
  (1.0322153568267822, 3)],
 'njet>=7': [(0, 0.9371899167696635),
  (0.9371899167696635, 1.04704749584198),
  (1.04704749584198, 3)]}

In [ ]:
# Load the saved model (e.g. epoch 45, check which one you want)
# Re-initialize model structure
model = DCTRNet(input_dim=x_tensor.shape[1], hidden_net=[64,32], 
                            hidden_net_classifier=[32,32], hidden_net_adv=[16,16]).to("cuda")

# Load state dict
# Assuming you want the one from adversarial training
model_path = "model_adv_epoch45.pth" 
model.load_state_dict(torch.load(model_path))
model.eval()

# Preparing full input data for inference
# We need to scale the full input data (dctr_inputs) using the same scaler fitted earlier
# dctr_inputs was defined earlier as np.stack(data).T
# scaler was fitted on x_numpy = dctr_inputs.astype(np.float32)

x_full_scaled = scaler.transform(dctr_inputs.astype(np.float32))
x_full_tensor = torch.from_numpy(x_full_scaled).to("cuda")

with torch.no_grad():
    predictions = model(x_full_tensor)
    # The model output is logits, so apply sigmoid
    predictions = torch.sigmoid(predictions).squeeze().cpu().numpy()

print(f"Predictions computed for {len(predictions)} events.")
print(predictions[:10])